# Frozen Lake MDP
Notebook organizado con las mismas secciones comentadas que el script original.


## Dependencias
Este entorno solo requiere NumPy para construir las tablas del MDP.


In [18]:
import numpy as np


## Definición de la clase base
Creamos un contenedor vacío para adjuntar cada método por separado.


In [19]:
class FrozenLakeMDP:
    '''Contenedor de métodos para el MDP Frozen Lake 4x4.'''
    pass


-------------------------
(0) Parámetros del entorno
-------------------------


In [20]:
def __init__(self, grid, gamma=0.9):
    # -------------------------
    # (0) Parámetros del entorno
    # -------------------------
    self.grid = grid
    self.gamma = gamma

    # Grid fijo 4x4 -> 16 estados
    self.n = 4
    self.S = self.n * self.n   # 16 estados (0..15)
    self.A = 4                 # 4 acciones (N, S, E, W)

    # -------------------------
    # (1) Acciones: N,S,E,W
    # Representamos acciones como números para facilitar numpy:
    # 0=N, 1=S, 2=E, 3=W
    # -------------------------
    self.actions = ["N", "S", "E", "W"]

    # delta (dr, dc) para moverse en el grid
    self.deltas = {
        0: (-1, 0),  # N
        1: ( 1, 0),  # S
        2: ( 0, 1),  # E
        3: ( 0,-1),  # W
    }

    # -------------------------
    # (2) Estocasticidad (hielo resbaladizo)
    # Si intento a, realmente puede ocurrir:
    #   - a (deseada) con 1/3
    #   - perpendicular 1 con 1/3
    #   - perpendicular 2 con 1/3
    #
    # OJO: tu slip_actions estaba mal: estabas quitando la acción deseada.
    # Aquí está correcto:
    # -------------------------
    self.slip_actions = {
        0: [0, 3, 2],  # intento N -> N, W, E
        1: [1, 2, 3],  # intento S -> S, E, W
        2: [2, 0, 1],  # intento E -> E, N, S
        3: [3, 1, 0],  # intento W -> W, S, N
    }
    self.slip_probs = np.array([1/3, 1/3, 1/3], dtype=np.float32)

    # -------------------------
    # (3) Identificar Goal y Holes (terminales)
    # - H: terminal, recompensa 0
    # - G: terminal, recompensa +1 al ENTRAR (cuando s' == G)
    # -------------------------
    self.terminal = np.zeros(self.S, dtype=bool)
    self.goal_state = None
    self.holes = set()

    for r in range(self.n):
        for c in range(self.n):
            ch = self.grid[r][c]
            s = self.to_state(r, c)
            if ch == "H":
                self.terminal[s] = True
                self.holes.add(s)
            elif ch == "G":
                self.terminal[s] = True
                self.goal_state = s

    if self.goal_state is None:
        raise ValueError("El grid debe contener una 'G' (Goal).")

    # -------------------------
    # (4) Crear tablas del MDP
    # T(s,a,s'): probabilidad de ir de s a s' ejecutando acción a
    # R(s,a,s'): recompensa por transición s->s'
    # -------------------------
    self.T = np.zeros((self.S, self.A, self.S), dtype=np.float32)
    self.R = np.zeros((self.S, self.A, self.S), dtype=np.float32)

    # Llenar T y R
    self.build_T()
    self.build_R()

    # -------------------------
    # (5) Validación rápida: T debe sumar 1 en cada (s,a)
    # -------------------------
    self.validate_T()

FrozenLakeMDP.__init__ = __init__
del __init__


---------------------------------------------------------
(Estados) Conversión coordenadas <-> estado
s = 4*fila + col
---------------------------------------------------------


In [21]:
def to_state(self, r, c):
    return r * self.n + c

FrozenLakeMDP.to_state = to_state
del to_state


---------------------------------------------------------
(Estados) Conversión coordenadas <-> estado
s = 4*fila + col
---------------------------------------------------------


In [22]:
def to_rc(self, s):
    return divmod(s, self.n)

FrozenLakeMDP.to_rc = to_rc
del to_rc


---------------------------------------------------------
Movimiento determinístico con bordes:
Si me salgo del grid, me quedo en el mismo estado.
---------------------------------------------------------


In [23]:
def move(self, s, a):
    r, c = self.to_rc(s)
    dr, dc = self.deltas[a]
    nr, nc = r + dr, c + dc

    if nr < 0 or nr >= self.n or nc < 0 or nc >= self.n:
        return s
    return self.to_state(nr, nc)

FrozenLakeMDP.move = move
del move


---------------------------------------------------------
Construcción de T(s,a,s') con resbalón 1/3-1/3-1/3
---------------------------------------------------------


In [24]:
def build_T(self):
    for s in range(self.S):
        for a in range(self.A):

            # Si s es terminal: absorbente (se queda en s con prob 1)
            if self.terminal[s]:
                self.T[s, a, s] = 1.0
                continue

            # Si no es terminal: considerar 3 acciones reales posibles
            for p, a_real in zip(self.slip_probs, self.slip_actions[a]):
                sp = self.move(s, a_real)
                self.T[s, a, sp] += p

FrozenLakeMDP.build_T = build_T
del build_T


---------------------------------------------------------
Construcción de R(s,a,s')
Recompensa = 1 SOLO cuando el estado siguiente es G.
En H y demás estados la recompensa es 0.
---------------------------------------------------------


In [25]:
def build_R(self):
    for s in range(self.S):
        for a in range(self.A):
            for sp in range(self.S):
                if sp == self.goal_state:
                    self.R[s, a, sp] = 1.0

FrozenLakeMDP.build_R = build_R
del build_R


---------------------------------------------------------
Validación: T[s,a,*] suma 1 para todo s,a
---------------------------------------------------------


In [26]:
def validate_T(self, tol=1e-6):
    for s in range(self.S):
        for a in range(self.A):
            total = float(self.T[s, a].sum())
            if abs(total - 1.0) > tol:
                raise ValueError(f"T[s={s},a={a}] no suma 1. Suma {total}")

FrozenLakeMDP.validate_T = validate_T
del validate_T


-------------------------
Prueba rápida (para mostrar avances)
-------------------------


In [29]:
if __name__ == "__main__":
    grid = [
        "SFFF",
        "FHFH",
        "FFFH",
        "HFFG"
    ]

    mdp = FrozenLakeMDP(grid)

    print("Goal state:", mdp.goal_state)
    print("Holes:", sorted(mdp.holes))

    # Ejemplo: desde Start (s=0), intento ir E (a=2)
    s0, aE = 0, 2
    nonzero = [(sp, p) for sp, p in enumerate(mdp.T[s0, aE]) if p > 0]
    print("Transiciones desde s=0 intentando E:")
    for sp, p in nonzero:
        print(f"  s'={sp} prob={p:.3f}")



Goal state: 15
Holes: [5, 7, 11, 12]
Transiciones desde s=0 intentando E:
  s'=0 prob=0.333
  s'=1 prob=0.333
  s'=4 prob=0.333
